# 08 — Oncology Subgraph Analysis: Full Graph vs Domain-Specific Subgraphs

**Project:** DDI-Graph-LLM  
**Research Question:** How does graph boundary selection affect DDI type prediction? Do features computed from the full DDI network outperform features from domain-specific subgraphs?

**Three-level comparison:**
```
Full Graph (191K edges, 1,701 drugs)
  └── Oncology Subgraph (≈22K edges, at least one oncology drug)
        ├── Liquid Cancer Subgraph (≈8K edges, hematological malignancies)
        └── Solid Cancer Subgraph (≈14K edges, solid tumors)
```

**Methods:** Random Forest and GNN (GCN) evaluated on each graph level


In [ ]:
import pandas as pd
import numpy as np
import re
import time
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score, 
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Define Oncology Drug Lists

We classify oncology drugs into two categories based on the cancer type they primarily treat:
- **Liquid cancer** (hematological): Leukemia, Lymphoma, Myeloma drugs
- **Solid cancer**: Breast, Lung, Colorectal, Melanoma, Renal, Prostate, Ovarian drugs


In [ ]:
# Liquid cancer drugs (hematological malignancies)
LIQUID_DRUGS = {
    "Imatinib", "Dasatinib", "Nilotinib", "Bosutinib", "Ponatinib",
    "Daunorubicin", "Idarubicin", "Cytarabine", "Mitoxantrone",
    "Ibrutinib", "Idelalisib", "Venetoclax",
    "Bortezomib", "Carfilzomib", "Ixazomib",
    "Thalidomide", "Lenalidomide", "Pomalidomide",
    "Vincristine", "Vinblastine", "Cyclophosphamide", "Chlorambucil",
    "Mechlorethamine", "Melphalan", "Busulfan",
    "Bleomycin", "Etoposide", "Procarbazine",
    "Ruxolitinib", "Baricitinib", "Tofacitinib",
    "Methotrexate", "Carmustine", "Lomustine", "Fludarabine",
}

# Solid tumor drugs
SOLID_DRUGS = {
    "Tamoxifen", "Fulvestrant", "Anastrozole", "Letrozole", "Exemestane",
    "Lapatinib", "Neratinib", "Tucatinib",
    "Palbociclib", "Ribociclib", "Abemaciclib",
    "Paclitaxel", "Docetaxel", "Doxorubicin", "Capecitabine",
    "Erlotinib", "Gefitinib", "Afatinib", "Osimertinib", "Icotinib",
    "Crizotinib", "Ceritinib", "Alectinib", "Brigatinib",
    "Vemurafenib", "Dabrafenib", "Encorafenib",
    "Fluorouracil", "Irinotecan", "Oxaliplatin", "Regorafenib",
    "Sorafenib", "Sunitinib", "Pazopanib", "Axitinib", "Cabozantinib", "Lenvatinib",
    "Everolimus", "Temsirolimus",
    "Cisplatin", "Carboplatin", "Olaparib", "Rucaparib", "Niraparib",
    "Bicalutamide", "Flutamide", "Enzalutamide", "Abiraterone",
    "Diethylstilbestrol", "Degarelix", "Goserelin", "Leuprolide",
    "Gemcitabine", "Temozolomide", "Dacarbazine", "Topotecan",
    "Ifosfamide", "Altretamine", "Estramustine", "Thiotepa",
    "Mitomycin", "Teniposide", "Vinorelbine", "Vandetanib", "Epirubicin",
}

ALL_ONCOLOGY = LIQUID_DRUGS | SOLID_DRUGS
print(f"Liquid cancer drugs defined: {len(LIQUID_DRUGS)}")
print(f"Solid cancer drugs defined: {len(SOLID_DRUGS)}")
print(f"All oncology drugs: {len(ALL_ONCOLOGY)}")


In [ ]:
# NLP label extraction patterns (from Phase 1)
PATTERNS = [
    (r"metabolism.*(?:increased|decreased)", "metabolism"),
    (r"active metabolites", "metabolism"),
    (r"serum concentration.*(?:increased|decreased|reduced)", "concentration"),
    (r"bioavailability.*(?:increased|decreased)", "concentration"),
    (r"protein binding", "concentration"),
    (r"risk or severity of adverse effects", "adverse_effects"),
    (r"risk or severity of.*(?:prolongation|bleeding|hypotension|hypertension|heart failure|hyperkalemia)", "adverse_effects"),
    (r"risk of a hypersensitivity", "adverse_effects"),
    (r"therapeutic efficacy.*(?:decreased|increased)", "efficacy"),
    (r"decrease effectiveness", "efficacy"),
    (r"decrease in the absorption", "absorption"),
    (r"absorption.*(?:increased|decreased)", "absorption"),
    (r"excretion rate", "excretion"),
    (r"(?:increase|decrease).*(?:activities|activity)", "activity"),
]

def extract_label(desc):
    for pattern, label in PATTERNS:
        if re.search(pattern, desc.lower()):
            return label
    return None


## 2. Build Subgraphs

For each level, we filter edges where **at least one drug** belongs to the target category.


In [ ]:
df_full = pd.read_csv("../data/drug_interactions.csv")
df_full['label'] = df_full['Interaction Description'].apply(extract_label)
df_full = df_full.dropna(subset=['label'])

all_drugs = set(df_full['Drug 1'].unique()) | set(df_full['Drug 2'].unique())

# Check which oncology drugs are in the dataset
liquid_found = LIQUID_DRUGS & all_drugs
solid_found = SOLID_DRUGS & all_drugs
onc_found = ALL_ONCOLOGY & all_drugs

# Build subsets
df_onc = df_full[(df_full['Drug 1'].isin(onc_found)) | (df_full['Drug 2'].isin(onc_found))].copy()
df_liquid = df_full[(df_full['Drug 1'].isin(liquid_found)) | (df_full['Drug 2'].isin(liquid_found))].copy()
df_solid = df_full[(df_full['Drug 1'].isin(solid_found)) | (df_full['Drug 2'].isin(solid_found))].copy()

datasets = {
    'Full Graph': df_full,
    'Oncology': df_onc,
    'Liquid Cancer': df_liquid,
    'Solid Cancer': df_solid,
}

print(f"{'Dataset':<20} {'Edges':>10} {'Drugs':>8} {'Onc drugs':>10}")
print("-" * 50)
for name, df in datasets.items():
    drugs = set(df['Drug 1']) | set(df['Drug 2'])
    onc_count = len(drugs & onc_found)
    print(f"{name:<20} {len(df):>10,} {len(drugs):>8,} {onc_count:>10}")


## 3. Label Distribution Comparison

How does the interaction type distribution shift across subgraphs?


In [ ]:
# Compute normalized label distributions
label_dists = {}
for name, df in datasets.items():
    dist = df['label'].value_counts(normalize=True).sort_index()
    label_dists[name] = dist

dist_df = pd.DataFrame(label_dists).fillna(0) * 100

# ── Presentation-ready grouped bar chart ──
fig, ax = plt.subplots(figsize=(14, 7))
labels_order = dist_df.index.tolist()
x = np.arange(len(labels_order))
width = 0.2
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, (name, color) in enumerate(zip(dist_df.columns, colors)):
    bars = ax.bar(x + i * width, dist_df[name], width, label=name, color=color, alpha=0.85)

ax.set_ylabel('Percentage (%)', fontsize=13)
ax.set_title('DDI Type Distribution: Full Graph vs Oncology Subgraphs', fontsize=15, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(labels_order, rotation=30, ha='right', fontsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../data/viz_label_distribution_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("Saved: viz_label_distribution_comparison.png")


## 4. Network Structure Comparison

In [ ]:
# Build NetworkX graphs and compute stats
graphs = {}
for name, df in datasets.items():
    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_edge(row['Drug 1'], row['Drug 2'], label=row['label'])
    graphs[name] = G

print(f"{'Metric':<25} {'Full Graph':>12} {'Oncology':>12} {'Liquid':>12} {'Solid':>12}")
print("=" * 75)

for metric_name, metric_fn in [
    ("Nodes", lambda G: G.number_of_nodes()),
    ("Edges", lambda G: G.number_of_edges()),
    ("Density", lambda G: f"{nx.density(G):.4f}"),
    ("Avg Degree", lambda G: f"{np.mean([d for _,d in G.degree()]):.1f}"),
    ("Median Degree", lambda G: f"{np.median([d for _,d in G.degree()]):.0f}"),
    ("Max Degree", lambda G: max(d for _,d in G.degree())),
    ("WCC", lambda G: nx.number_weakly_connected_components(G)),
]:
    vals = [metric_fn(graphs[name]) for name in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']]
    print(f"{metric_name:<25} {str(vals[0]):>12} {str(vals[1]):>12} {str(vals[2]):>12} {str(vals[3]):>12}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for ax, (name, G), color in zip(axes, graphs.items(), colors):
    degs = [d for _, d in G.degree()]
    ax.hist(degs, bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Degree')
    ax.axvline(np.median(degs), color='black', linestyle='--', alpha=0.6,
               label=f'Median: {np.median(degs):.0f}')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Count')
plt.suptitle('Degree Distributions Across Graph Levels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../data/viz_degree_distributions.png", dpi=200, bbox_inches='tight')
plt.show()


## 5. Feature Engineering Pipeline

Same 14-feature pipeline as Phase 2, applied independently to each subgraph.


In [ ]:
FEATURE_COLS = [
    "out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u",
    "out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v",
    "common_neighbors", "jaccard", "same_community", "degree_diff",
]

def compute_features(G):
    """Compute all node and edge features for a graph."""
    G_und = G.to_undirected()
    
    # Node features
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    betweenness = nx.betweenness_centrality(G, k=min(100, G.number_of_nodes()), seed=42)
    clustering = nx.clustering(G_und)
    pagerank = nx.pagerank(G)
    
    # Communities
    from networkx.algorithms.community import louvain_communities
    comms = louvain_communities(G_und, seed=42)
    comm_map = {}
    for idx, comm in enumerate(comms):
        for node in comm:
            comm_map[node] = idx
    
    # Neighbor sets for fast pairwise computation
    neighbor_sets = {n: set(G_und.neighbors(n)) for n in G_und.nodes()}
    
    # Edge features
    records = []
    for u, v, data in G.edges(data=True):
        cn = len(neighbor_sets[u] & neighbor_sets[v])
        union = len(neighbor_sets[u] | neighbor_sets[v])
        jc = cn / union if union > 0 else 0.0
        sc = int(comm_map.get(u, -1) == comm_map.get(v, -2))
        
        records.append({
            "drug_u": u, "drug_v": v, "label": data.get("label"),
            "out_degree_u": out_deg[u], "in_degree_u": in_deg[u],
            "betweenness_u": betweenness[u], "clustering_u": clustering[u],
            "pagerank_u": pagerank[u],
            "out_degree_v": out_deg[v], "in_degree_v": in_deg[v],
            "betweenness_v": betweenness[v], "clustering_v": clustering[v],
            "pagerank_v": pagerank[v],
            "common_neighbors": cn, "jaccard": jc,
            "same_community": sc,
            "degree_diff": abs((in_deg[u]+out_deg[u]) - (in_deg[v]+out_deg[v])),
        })
    
    return pd.DataFrame(records), len(comms)


def train_eval_rf(df_feat, name=""):
    """Train RF and return macro-F1 + per-class F1."""
    X = df_feat[FEATURE_COLS].values
    y = df_feat['label'].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    clf = RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=42,
        class_weight="balanced", n_jobs=-1
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    per_class = {}
    for label in sorted(set(y)):
        mask = y_test == label
        if mask.sum() > 0:
            per_class[label] = f1_score(y_test == label, y_pred == label)
    
    importances = dict(zip(FEATURE_COLS, clf.feature_importances_))
    
    return {
        'macro_f1': macro_f1,
        'y_test': y_test, 'y_pred': y_pred,
        'per_class': per_class,
        'importances': importances,
        'clf': clf,
    }


In [ ]:
class GCNEdgeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels, num_classes),
        )
    
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index)
        src, dst = edge_index
        edge_repr = torch.cat([h[src], h[dst]], dim=1)
        return self.edge_mlp(edge_repr)


def train_eval_gnn(G, df_feat, name="", epochs=200):
    """Train GCN and return macro-F1."""
    # Map drugs to IDs
    all_drugs_g = sorted(G.nodes())
    drug_to_id = {d: i for i, d in enumerate(all_drugs_g)}
    num_nodes = len(all_drugs_g)
    
    # Edge index
    src_ids = df_feat['drug_u'].map(drug_to_id).values
    dst_ids = df_feat['drug_v'].map(drug_to_id).values
    edge_index = torch.tensor(np.stack([src_ids, dst_ids]), dtype=torch.long)
    
    # Edge labels
    le = LabelEncoder()
    edge_labels = torch.tensor(le.fit_transform(df_feat['label'].values), dtype=torch.long)
    num_classes = len(le.classes_)
    
    # Node features from edge features (average)
    node_feat_cols_u = ["out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u"]
    node_feat_cols_v = ["out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v"]
    
    node_features = np.zeros((num_nodes, 5))
    node_counts = np.zeros(num_nodes)
    for _, row in df_feat.iterrows():
        uid = drug_to_id[row['drug_u']]
        vid = drug_to_id[row['drug_v']]
        node_features[uid] += row[node_feat_cols_u].values.astype(float)
        node_counts[uid] += 1
        node_features[vid] += row[node_feat_cols_v].values.astype(float)
        node_counts[vid] += 1
    node_counts[node_counts == 0] = 1
    node_features /= node_counts[:, None]
    
    scaler = MinMaxScaler()
    node_features = scaler.fit_transform(node_features)
    x = torch.tensor(node_features, dtype=torch.float)
    
    # Train/test split on edges
    n_edges = len(df_feat)
    indices = np.arange(n_edges)
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=42, stratify=df_feat['label'].values
    )
    train_mask = torch.zeros(n_edges, dtype=torch.bool)
    test_mask = torch.zeros(n_edges, dtype=torch.bool)
    train_mask[train_idx] = True
    test_mask[test_idx] = True
    
    # Data object
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_labels, num_nodes=num_nodes).to(device)
    
    # Class weights
    train_labels = edge_labels[train_mask].numpy()
    class_counts = Counter(train_labels)
    n_train = len(train_labels)
    weights = torch.tensor(
        [n_train / (num_classes * class_counts.get(i, 1)) for i in range(num_classes)],
        dtype=torch.float
    ).to(device)
    
    # Model
    model = GCNEdgeClassifier(5, 64, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Train
    train_mask_d = train_mask.to(device)
    test_mask_d = test_mask.to(device)
    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_mask_d], data.edge_attr[train_mask_d])
        loss.backward()
        optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out[test_mask_d].argmax(dim=1).cpu().numpy()
        true = data.edge_attr[test_mask_d].cpu().numpy()
    
    macro_f1 = f1_score(true, pred, average='macro', zero_division=0)
    
    true_labels = le.inverse_transform(true)
    pred_labels = le.inverse_transform(pred)
    
    return {
        'macro_f1': macro_f1,
        'y_test': true_labels,
        'y_pred': pred_labels,
        'le': le,
    }


## 6. Run Experiments

Compute features and train RF + GNN on each graph level.


In [ ]:
results = {}

for name, G in graphs.items():
    print(f"\n{'='*60}")
    print(f"Processing: {name}")
    print(f"{'='*60}")
    
    # Compute features
    t0 = time.time()
    df_feat, n_comms = compute_features(G)
    print(f"  Features computed: {len(df_feat):,} edges, {n_comms} communities, {time.time()-t0:.1f}s")
    
    # RF
    t0 = time.time()
    rf_result = train_eval_rf(df_feat, name)
    print(f"  RF Macro-F1: {rf_result['macro_f1']:.4f} ({time.time()-t0:.1f}s)")
    
    # GNN
    t0 = time.time()
    gnn_result = train_eval_gnn(G, df_feat, name, epochs=200)
    print(f"  GNN Macro-F1: {gnn_result['macro_f1']:.4f} ({time.time()-t0:.1f}s)")
    
    results[name] = {
        'df_feat': df_feat,
        'rf': rf_result,
        'gnn': gnn_result,
    }

print(f"\n{'='*60}")
print("ALL EXPERIMENTS COMPLETE")
print(f"{'='*60}")


## 7. Results Summary

In [ ]:
print(f"{'Graph Level':<20} {'RF Macro-F1':>12} {'GNN Macro-F1':>13}")
print("=" * 47)
for name in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']:
    rf_f1 = results[name]['rf']['macro_f1']
    gnn_f1 = results[name]['gnn']['macro_f1']
    print(f"{name:<20} {rf_f1:>12.4f} {gnn_f1:>13.4f}")


### 7.1 Main Result: RF vs GNN Across Graph Levels

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

graph_names = ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']
rf_scores = [results[n]['rf']['macro_f1'] for n in graph_names]
gnn_scores = [results[n]['gnn']['macro_f1'] for n in graph_names]

x = np.arange(len(graph_names))
width = 0.35

bars1 = ax.bar(x - width/2, rf_scores, width, label='Random Forest', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, gnn_scores, width, label='GNN (GCN)', color='#9b59b6', edgecolor='white')

# Add value labels
for bar, score in zip(bars1, rf_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar, score in zip(bars2, gnn_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Macro-F1 Score', fontsize=13)
ax.set_title('DDI Type Prediction: Full Graph vs Domain-Specific Subgraphs', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(graph_names, fontsize=12)
ax.legend(fontsize=12, loc='upper right')
ax.set_ylim(0, 1.1)
ax.axhline(y=1/7, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../data/viz_main_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("Saved: viz_main_comparison.png")


### 7.2 RF Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
cmaps = ['Blues', 'Greens', 'Reds', 'Oranges']

for ax, (name, cmap) in zip(axes.flat, zip(graph_names, cmaps)):
    r = results[name]['rf']
    labels = sorted(set(r['y_test']))
    cm = confusion_matrix(r['y_test'], r['y_pred'], labels=labels, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap=cmap, values_format='.2f', colorbar=False)
    ax.set_title(f'RF: {name}', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('Normalized Confusion Matrices (Random Forest)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("../data/viz_rf_confusion_matrices.png", dpi=200, bbox_inches='tight')
plt.show()


### 7.3 Feature Importance: How It Shifts Across Graph Levels

In [ ]:
# Build feature importance comparison
imp_data = {}
for name in graph_names:
    imp_data[name] = results[name]['rf']['importances']

imp_df = pd.DataFrame(imp_data)

fig, ax = plt.subplots(figsize=(14, 8))
imp_df_plot = imp_df.sort_values('Full Graph', ascending=True)
x = np.arange(len(imp_df_plot))
width = 0.2

for i, (name, color) in enumerate(zip(graph_names, ['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])):
    ax.barh(x + i * width, imp_df_plot[name], width, label=name, color=color, alpha=0.85)

ax.set_yticks(x + width * 1.5)
ax.set_yticklabels(imp_df_plot.index, fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('RF Feature Importance Across Graph Levels', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')

plt.tight_layout()
plt.savefig("../data/viz_feature_importance.png", dpi=200, bbox_inches='tight')
plt.show()


### 7.4 Graph Hierarchy Overview (for Presentation)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Summary stats as table-like visualization
categories = ['Full Graph', 'Oncology\nSubgraph', 'Liquid\nCancer', 'Solid\nCancer']
edges = [len(datasets[n]) for n in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']]
nodes = [len(set(datasets[n]['Drug 1']) | set(datasets[n]['Drug 2'])) 
         for n in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']]
rf_f1s = [results[n]['rf']['macro_f1'] for n in graph_names]

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(categories, edges, color=colors, alpha=0.8, edgecolor='white', linewidth=2)

for bar, n, e, f1 in zip(bars, nodes, edges, rf_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{e:,} edges\n{n:,} drugs\nRF F1={f1:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Number of Edges', fontsize=13)
ax.set_title('DDI Network: Graph Hierarchy Overview', fontsize=15, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../data/viz_graph_hierarchy.png", dpi=200, bbox_inches='tight')
plt.show()


## 8. Liquid vs Solid Cancer: Detailed Comparison

Do RF and GNN behave differently on these two cancer subtypes?


In [ ]:
# Per-class F1 comparison: Liquid vs Solid
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, method, color1, color2 in [
    (axes[0], 'rf', '#e74c3c', '#f39c12'),
    (axes[1], 'gnn', '#e74c3c', '#f39c12'),
]:
    r_liq = results['Liquid Cancer'][method]
    r_sol = results['Solid Cancer'][method]
    
    all_labels = sorted(set(r_liq['y_test']) | set(r_sol['y_test']))
    
    liq_f1 = f1_score(r_liq['y_test'], r_liq['y_pred'], labels=all_labels, average=None, zero_division=0)
    sol_f1 = f1_score(r_sol['y_test'], r_sol['y_pred'], labels=all_labels, average=None, zero_division=0)
    
    x = np.arange(len(all_labels))
    width = 0.35
    ax.bar(x - width/2, liq_f1, width, label='Liquid Cancer', color=color1, alpha=0.8)
    ax.bar(x + width/2, sol_f1, width, label='Solid Cancer', color=color2, alpha=0.8)
    
    method_name = 'Random Forest' if method == 'rf' else 'GNN (GCN)'
    ax.set_title(f'{method_name}: Per-Class F1', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('F1 Score')
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Liquid vs Solid Cancer: Per-Class Performance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../data/viz_liquid_vs_solid_perclass.png", dpi=200, bbox_inches='tight')
plt.show()


## 9. Save Results

In [ ]:
# Save feature dataframes
for name in graph_names:
    safe_name = name.lower().replace(' ', '_')
    results[name]['df_feat'].to_csv(f"../data/edge_features_{safe_name}.csv", index=False)
    print(f"Saved: edge_features_{safe_name}.csv")

# Save summary
summary = []
for name in graph_names:
    summary.append({
        'graph': name,
        'edges': len(results[name]['df_feat']),
        'rf_macro_f1': results[name]['rf']['macro_f1'],
        'gnn_macro_f1': results[name]['gnn']['macro_f1'],
    })
pd.DataFrame(summary).to_csv("../data/experiment_summary.csv", index=False)
print("Saved: experiment_summary.csv")


## 10. Summary & Key Takeaways

### Results Table

| Graph Level | Edges | RF Macro-F1 | GNN Macro-F1 |
|-------------|-------|-------------|--------------|
| Full Graph | 191K | TBD | TBD |
| Oncology Subgraph | ~22K | TBD | TBD |
| Liquid Cancer | ~8K | TBD | TBD |
| Solid Cancer | ~14K | TBD | TBD |

### Key Questions Answered

1. **Does graph boundary affect prediction?** Compare Full vs Oncology RF/GNN scores.
2. **Does RF still dominate on smaller graphs?** Check if GNN closes the gap on smaller subgraphs.
3. **Do Liquid and Solid cancer have different DDI patterns?** Compare label distributions and per-class F1.
4. **Which features matter most in each domain?** Compare feature importance rankings.

### Visualizations Saved (for Presentation)
- `viz_label_distribution_comparison.png` — How DDI types shift across domains
- `viz_degree_distributions.png` — Network structure comparison
- `viz_main_comparison.png` — Core result: RF vs GNN across graph levels
- `viz_rf_confusion_matrices.png` — Where each model makes mistakes
- `viz_feature_importance.png` — Which features matter in each domain
- `viz_graph_hierarchy.png` — Overview slide for presentation
- `viz_liquid_vs_solid_perclass.png` — Cancer subtype comparison
